In [1]:
# Import libraries
from pathlib import Path
import numpy as np
import sys
sys.path.append('../')
import DataLoader as dldr
import AcquisitionFunctions as af
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Kernel, RBF, RationalQuadratic, Matern, ExpSineSquared
from scipy.stats import norm

In [2]:
# Set root directory to load data-points, week & function numbers.
rootDir: Path = Path('..')
weekNbr: int = 1
funcNbr: int = 4

In [3]:
X_inputs_until_prev_week = dldr.load_cumulative_inputs(rootDir, weekNbr - 1, funcNbr)
Y_outputs_until_prev_week = dldr.load_cumulative_outputs(rootDir, weekNbr - 1, funcNbr)

# Print and check some contents from the parsed input & output arrays.
print(f"Function {funcNbr}: until previous week has {len(X_inputs_until_prev_week)} input data-points and {len(Y_outputs_until_prev_week)} output values.")

Week: 0, function 4: Combined initial & weekly samples contain 30 input data-points.
Week: 0, function 4: Combined initial & weekly outputs contain 30 output values, with maximum = -4.025542281908162.
Function 4: until previous week has 30 input data-points and 30 output values.


In [4]:
y_max_prev_wk = np.max(Y_outputs_until_prev_week)
# Load this week's data
X_this_week_inputs = dldr.load_inputs(rootDir, weekNbr, funcNbr)
Y_this_week_output = dldr.load_output(rootDir, weekNbr, funcNbr)

In [5]:
print(f"Function {funcNbr}: until previous week input data-points: \n{X_inputs_until_prev_week}")
print(f"Function {funcNbr}: until previous week output values: \n{Y_outputs_until_prev_week}")
print(f"Function {funcNbr}: this week's input data-point: {X_this_week_inputs}")
print(f"Function {funcNbr}: this week's output value: {Y_this_week_output}")

Function 4: until previous week input data-points: 
[[0.89698105 0.72562797 0.17540431 0.70169437]
 [0.8893564  0.49958786 0.53926886 0.50878344]
 [0.25094624 0.03369313 0.14538002 0.49493242]
 [0.34696206 0.0062504  0.76056361 0.61302356]
 [0.12487118 0.12977019 0.38440048 0.2870761 ]
 [0.80130271 0.50023109 0.70664456 0.19510284]
 [0.24770826 0.06044543 0.04218635 0.44132425]
 [0.74670224 0.7570915  0.36935306 0.20656628]
 [0.40066503 0.07257425 0.88676825 0.24384229]
 [0.6260706  0.58675126 0.43880578 0.77885769]
 [0.95713529 0.59764438 0.76611385 0.77620991]
 [0.73281243 0.14524998 0.47681272 0.13336573]
 [0.65511548 0.07239183 0.68715175 0.08151656]
 [0.21973443 0.83203134 0.48286416 0.08256923]
 [0.48859419 0.2119651  0.93917791 0.37619173]
 [0.16713049 0.87655456 0.21723954 0.95980098]
 [0.21691119 0.16608583 0.24137226 0.77006248]
 [0.38748784 0.80453226 0.75179548 0.72382744]
 [0.98562189 0.66693268 0.15678328 0.8565348 ]
 [0.03782483 0.66485335 0.16198218 0.25392378]
 [0.6834

In [6]:
# Combine cumulative inputs until previous week and this week's inputs.
X_inputs_combined = X_inputs_until_prev_week.copy()
X_inputs_combined = np.append(X_inputs_combined, np.array([X_this_week_inputs]), axis=0)

# Combine cumulative outputs until previous week and this week's output.
Y_outputs_combined = Y_outputs_until_prev_week.copy()
Y_outputs_combined = np.append(Y_outputs_combined, np.array([Y_this_week_output]), axis=0)

In [7]:
print(f"Samples aggregated till this week contain {len(X_inputs_combined)} input data-points.")

Samples aggregated till this week contain 31 input data-points.


In [8]:
print(f"Existing (till last week) maximum output: {y_max_prev_wk}, this week's output: {Y_this_week_output}.")

if Y_this_week_output <= y_max_prev_wk:
    print('New output less than existing maximum, do more exploration.')
else:
    print('New output greater than existing maximum, continue exploitation.') 

Existing (till last week) maximum output: -4.025542281908162, this week's output: -3.985750969810286.
New output greater than existing maximum, continue exploitation.


In [9]:
# Compute the maximum output value from the aggregated (till this week) data-set.
y_curr_max = np.max(Y_outputs_combined)
print(f"Aggregated (till this week) data-set contain {len(Y_outputs_combined)} output values, with maximum = {y_curr_max}.")

Aggregated (till this week) data-set contain 31 output values, with maximum = -3.985750969810286.


In [10]:
# Initialize a Gaussian Process surrogate model, which will work as a proxy to
# the (unknown) actual function.
kernel: Kernel = RBF(length_scale=1.0, length_scale_bounds='fixed')
model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)
model.fit(X_inputs_combined, Y_outputs_combined)

,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".",RBF(length_scale=1)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-10
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",9
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",False
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `."

In [11]:
# Apply the model against points on an evaluation grid and capture the predicted values.
x_grid = np.linspace(0, 1, 2000).reshape(-1, 4)
y_pred_means, y_pred_sigmas = model.predict(x_grid, return_std=True)

In [12]:
print(f"x_grid.shape: {x_grid.shape}, y_pred_means.shape: {y_pred_means.shape}, y_pred_sigmas.shape: {y_pred_sigmas.shape}")

x_grid.shape: (500, 4), y_pred_means.shape: (500,), y_pred_sigmas.shape: (500,)


In [13]:
# Initialize an acquisition function to choose the next data-point on the grid to search.
confid_intvl = 0.80
eta = 0.01
acquisition_function = af.ucb(confid_intvl, y_pred_means, y_pred_sigmas)
#acquisition_function = af.prob_improvement(eta, y_pred_means, y_pred_sigmas, y_curr_max)
af_max_idx = np.argmax(acquisition_function)
af_max = acquisition_function[af_max_idx]

x_next = x_grid[af_max_idx]
y_next_mean_pred = y_pred_means[af_max_idx]
y_next_sigma_pred = y_pred_sigmas[af_max_idx]
print('af_max: ', np.max(acquisition_function))
print(f"acquisition_function max: {af_max}, index of max: {af_max_idx}")
print(f"y_pred_means[{af_max_idx}]: {y_next_mean_pred}, y_pred_sigmas[{af_max_idx}]: {y_next_sigma_pred}")

# The relationship between af_max, y_next_mean_pred, y_next_sigma_pred 
# & y_curr_max follows the equation from the function prob_improvement()
# in the module: AcquisitionFunctions. This has been verified separately.

af_max:  -1.484425130714848
acquisition_function max: -1.484425130714848, index of max: 197
y_pred_means[197]: -1.4935415946297326, y_pred_sigmas[197]: 0.007113614590303738


In [14]:
print(f"Current maximum output: {y_curr_max}")
print(f"Next data-point to search: {x_next}.")

Current maximum output: -3.985750969810286
Next data-point to search: [0.3941971  0.39469735 0.3951976  0.39569785].
